# Practice 36 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — iris: vectorized columns

Load `sns.load_dataset('iris')`.

1. Add `petal_ratio` (petal_length / petal_width) and `sepal_ratio` (sepal_length / sepal_width) — vectorized, no `.apply()`.
2. Use `np.select()` to add `petal_class`: `'long'` (petal_length > 5), `'medium'` (petal_length > 3), `'short'` otherwise.
3. Convert `species` to `category`. Compare memory usage with `.memory_usage(deep=True)` before and after.
4. Which species has the highest mean `petal_ratio`? Which has the lowest? Use whatever method you like.

In [10]:
# Your code here

iris = sns.load_dataset('iris')
iris['petal_ratio'] = iris['petal_length']/iris['petal_width']
iris['sepal_ratio'] = iris['sepal_length']/iris['sepal_width']

condis = [iris['petal_length']>5 , iris['petal_length']>3]
choi = ['long','medium']
iris['petal_class'] = np.select(condis, choi, default='short')

print(iris.memory_usage(deep=True).sum())
iris['species'] = iris['species'].astype('category')
print(iris.memory_usage(deep=True).sum())

g = iris.groupby('species', observed=True)['petal_ratio'].mean()

print(g.idxmax())
print(g.idxmin())



26443
17097
setosa
virginica


---
## Level 2 — rewrite challenge

Below is a small employee dataset and five snippets using `.apply()`. Rewrite each one without `.apply()`. Then use `%timeit` to compare the speed of snippet **C** (apply vs your version).

```python
# A
df['annual'] = df.apply(lambda row: row['salary'] * 12, axis=1)

# B
df['senior'] = df['years'].apply(lambda x: True if x >= 8 else False)

# C
df['band'] = df['salary'].apply(
    lambda x: 'exec' if x > 9000
    else 'senior' if x > 6000
    else 'mid' if x > 3500
    else 'junior')

# D
df['tag'] = df.apply(lambda row: row['dept'] + '-' + row['level'], axis=1)

# E
df['log_salary'] = df['salary'].apply(lambda x: np.log(x))
```

After rewriting, collect the names of all `'exec'` band employees into a plain Python list using a list comprehension on `df`.

In [17]:
df = pd.DataFrame({
    'name':   ['Alice','Bob','Carol','David','Eva','Frank','Grace','Hiro','Ivy','Jack'],
    'dept':   ['Eng','Eng','HR','HR','Eng','Sales','Sales','Eng','HR','Sales'],
    'level':  ['L3','L5','L2','L4','L6','L3','L5','L4','L3','L6'],
    'salary': [3200, 7100, 2900, 4800, 11000, 3600, 6500, 5200, 3100, 9500],
    'years':  [2, 9, 1, 5, 14, 3, 8, 6, 2, 12]
})

# Your rewrites here
#A 
df['annual'] = df['salary']*12
#B
df['senior']= df['years']>=8
#C
df['band'] = pd.cut(df['salary'], bins = [0,3500,6000,9000, np.inf], labels=['junior','mid','senior','exec'], right=True)
#D
df['tag'] = df['dept'] + "-" + df['level']
#E
df['log_salary'] = np.log(df['salary'])


[name for name in df[df['band']=='exec']['name']]


['Eva', 'Jack']

---
## Level 3 — diamonds: open questions

Load `sns.load_dataset('diamonds')`. No `.apply()` anywhere in this task.

Answer all five questions:

1. What fraction of diamonds are both above the median `carat` and above the median `price`? (Single vectorized expression.)
2. Add `price_per_carat` vectorized. Which `cut` has the highest mean `price_per_carat`? Does the answer surprise you given the raw price ranking?
3. Use `np.select()` to bucket diamonds into `'small'` (carat < 0.5), `'mid'` (0.5–1.5), `'large'` (> 1.5). What is the mean `price_per_carat` for each bucket?
4. Convert `cut`, `color`, and `clarity` to `category`. How much memory (in MB) does the DataFrame use before vs after?
5. `np.corrcoef` between `np.log(price)` and `carat` — what is the correlation? Is it stronger or weaker than the raw `price` vs `carat` correlation?

In [30]:
# Your code here

diamonds = sns.load_dataset('diamonds')

((diamonds['carat']>diamonds['carat'].mean()) & (diamonds['price']>diamonds['price'].mean())).mean()

diamonds['price_per_carat'] = diamonds['price']/diamonds['carat']
diamonds.groupby('cut', observed=True)['price_per_carat'].mean().idxmax()
diamonds.groupby('cut', observed=True)['price'].mean().idxmax()

conds = [diamonds['carat']<0.5, diamonds['carat']<1.5]
choi = ['small','mid']
diamonds['sizeb'] = np.select(conds, choi, default='large')

diamonds.groupby('sizeb')['price_per_carat'].mean()

print(diamonds.memory_usage(deep=True).sum())
diamonds[['cut','color','clarity']] = diamonds[['cut','color','clarity']].astype('category')
print(diamonds.memory_usage(deep=True).sum())
diamonds['log_price'] = np.log(diamonds['price'])

print(np.corrcoef(diamonds['log_price'], diamonds['carat'])[0,1])


print(np.corrcoef(diamonds['price'], diamonds['carat'])[0,1])



6900298
6900298
0.9202065980266586
0.921591301193477


---
## Level 4 — sales pipeline

Write a `.pipe()` chain with three functions on the sales data below. No `.apply()` allowed.

- **`clean(df)`** — convert `region` and `category` to `category` dtype; add `revenue` (units × price) vectorized
- **`flag(df)`** — add `above_avg_rev`: True where `revenue` is above the overall mean. Add `margin_class` using `np.select()`: `'high'` (margin > 0.4), `'mid'` (margin > 0.2), `'low'` otherwise
- **`summarize(df)`** — return a pivot table: mean `revenue` by `region` × `margin_class`, with `observed=True`

After the chain:
- Stack the pivot table so you have a Series indexed by `(region, margin_class)`. What is the single highest-revenue combination?

In [37]:
sales = pd.DataFrame({
    'region':   ['North','North','North','South','South','South','East','East','East','West','West','West'],
    'category': ['Electronics','Clothing','Food','Electronics','Clothing','Food',
                 'Electronics','Clothing','Food','Electronics','Clothing','Food'],
    'units':    [120, 340, 890, 95, 410, 1020, 200, 280, 760, 155, 390, 640],
    'price':    [450, 35, 8, 420, 40, 7, 460, 38, 9, 430, 36, 8],
    'margin':   [0.45, 0.22, 0.12, 0.41, 0.25, 0.10, 0.50, 0.18, 0.15, 0.38, 0.30, 0.11]
})

# Your code here

def clean(df):
    df[['region','category']] = df[['region','category']].astype('category')
    df['revenue'] = df['units']* df['price']
    return df 
def flag(df):
    df['above_avg_rev'] = df['revenue']>df['revenue'].mean()
    conds = [df['margin']>0.4, df['margin']>0.2]
    chois = ['high', 'mid']
    df['margin_class'] = np.select(conds, chois, default = 'low')
    return df 

def summarize(df):
    p = df.pivot_table(
        index = 'region',
        columns = 'margin_class',
        values = 'revenue',
        observed = True
    )
    return p


res = sales.copy().pipe(clean).pipe(flag).pipe(summarize)
res.stack()


res.stack().idxmax()


('East', 'high')